# Réplicas de MD (parte 1 de 2) — System_2A y System_2B
Corre 2 réplicas adicionales por sistema (r2, r3; la r1 es la corrida original) con **el mismo protocolo**: parten del sistema ya minimizado y solo cambia la semilla de las velocidades del NVT. Cada una: NVT 100 ps → NPT 500 ps → producción 1 ns → corrección de PBC.

**Antes:** *Entorno de ejecución > Cambiar tipo > A100*. Ejecuta en orden (la celda de conda reinicia el entorno: es normal).
**Sube** `D_F_energia/replicas_inputs.zip` cuando lo pida. Al final descarga `replicas_md.zip` y úsalo en el notebook 2 (MM/GBSA).

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Instalar conda
El entorno se reinicia solo al terminar y aparece un aviso de sesión caída: es normal. Sigue con la celda 3.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## 3. Instalar GROMACS con CUDA

In [ ]:
!mamba install -y -c conda-forge -c bioconda --no-pin gromacs=*=*cuda* cuda-version=12.6
!ln -sf /usr/local/bin/gmx_mpi /usr/bin/gmx
!ln -sf /usr/local/bin/gmx_mpi /usr/local/bin/gmx
!gmx --version | grep -E "GROMACS version|GPU support"

## Subir entradas

In [ ]:
import os, subprocess
from google.colab import files
IN='/content/in'; OUT='/content/replicas'
os.makedirs(IN,exist_ok=True); os.makedirs(OUT,exist_ok=True)
if not os.path.exists(f'{IN}/replica.sh'):
    subido=files.upload()
    subprocess.run(['unzip','-o',list(subido)[0],'-d',IN],check=True)
!ls {IN} {IN}/2A

## Correr las 4 réplicas (2A r2, 2A r3, 2B r2, 2B r3)
Cada una tarda pocos minutos en A100. Si se desconecta, vuelve a ejecutar desde la celda de conda: las terminadas se saltan solo si /content sigue; si no, empieza de nuevo.

In [ ]:
%%bash
cd /content
# semillas fijas y distintas para que sea reproducible
for SIS in 2A 2B; do
  bash /content/in/replica.sh $SIS 2 1002 /content/in /content/replicas 2>&1 | grep -E "^===|Performance|Fatal|ERROR|Temperature|Density|Pressure"
  bash /content/in/replica.sh $SIS 3 1003 /content/in /content/replicas 2>&1 | grep -E "^===|Performance|Fatal|ERROR|Temperature|Density|Pressure"
done

## Empaquetar y descargar
Solo lo necesario para MM/GBSA (≈ 55 MB).

In [ ]:
%%bash
cd /content/replicas
rm -f ../replicas_md.zip
zip -qr ../replicas_md.zip */proteina.tpr */proteina.xtc */proteina.ndx */topol*.top */topol_Protein_chain_*.itp */posre_Protein_chain_*.itp */B8_*.xvg */md.log
ls -lh ../replicas_md.zip

In [ ]:
from google.colab import files
files.download('/content/replicas_md.zip')